# ENTSO-E Parquet Data Check (NL)

This notebook loads the ENTSO-E prices stored in Parquet, checks the latest timestamp,
and optionally triggers the daily updater for the last 2 days to top up the dataset.
It is placed next to `app.py` for quick verification during development.

In [1]:
# Optional: ensure dependencies
%pip -q install pandas pyarrow matplotlib

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os, sys, datetime as dt, pandas as pd, pathlib, pytz

# Parameters
ZONE = os.getenv('WB4U_ZONE', '10YNL----------L')
CONTRACT = os.getenv('WB4U_CONTRACT', 'A01')  # Day-ahead
COUNTRY_TAG = os.getenv('WB4U_COUNTRY_TAG', 'NL')

# Where Parquet lives (single file and partitioned dataset)
PARQUET_FILE = os.getenv('WB4U_PARQUET_FILE', 'entsoe_prices_NL.parquet')
PARQUET_DIR = os.getenv('WB4U_PARQUET_DIR', 'data/entsoe/parquet')

def find_repo_root():
    if ('entsoe_daily_update.py').exists() or (q / '.git').exists():
            return q
    return pathlib.Path.cwd()

REPO_ROOT = find_repo_root()
PARQUET_FILE = str((REPO_ROOT / PARQUET_FILE).resolve()) if not os.path.isabs(PARQUET_FILE) else PARQUET_FILE
PARQUET_DIR = str((REPO_ROOT / PARQUET_DIR).resolve()) if not os.path.isabs(PARQUET_DIR) else PARQUET_DIR

print('Config ->', dict(zone=ZONE, contract=CONTRACT, file=PARQUET_FILE, dir=PARQUET_DIR))

Config -> {'zone': '10YNL----------L', 'contract': 'A01', 'file': 'C:\\Users\\20203525\\Documents\\2025 2026\\WB4U\\energy-profile-HEMS\\data\\entsoe\\entsoe_prices_NL.parquet', 'dir': 'C:\\Users\\20203525\\Documents\\2025 2026\\WB4U\\energy-profile-HEMS\\data\\entsoe\\parquet'}


In [3]:
# Load helpers (single file or partitioned dataset)
import pyarrow.dataset as ds

def load_from_file(path: str, zone: str, contract: str) -> pd.DataFrame:
    if not os.path.exists(path):
        return pd.DataFrame(columns=['bidding_zone','contract','ts_utc','price','currency','unit','resolution'])
    df = pd.read_parquet(path)
    if df.empty:
        return df
    df['ts_utc'] = pd.to_datetime(df['ts_utc'], utc=True)
    df = df[(df['bidding_zone']==zone) & (df['contract']==contract)]
    return df.sort_values('ts_utc').reset_index(drop=True)

def load_from_dataset(root: str, zone: str, contract: str) -> pd.DataFrame:
    if not os.path.isdir(root):
        return pd.DataFrame(columns=['bidding_zone','contract','ts_utc','price','currency','unit','resolution'])
    dataset = ds.dataset(root, format='parquet', partitioning='hive')
    filt = (ds.field('bidding_zone') == zone) & (ds.field('contract') == contract)
    cols = ['bidding_zone','contract','ts_utc','price','currency','unit','resolution']
    try:
        tbl = dataset.to_table(filter=filt, columns=cols)
    except Exception:
        # Fallback: load without filter
        tbl = dataset.to_table(columns=cols)
    df = tbl.to_pandas()
    if df.empty:
        return df
    df['ts_utc'] = pd.to_datetime(df['ts_utc'], utc=True)
    return df.sort_values('ts_utc').reset_index(drop=True)

def load_prices(zone: str, contract: str) -> pd.DataFrame:
    df = load_from_file(PARQUET_FILE, zone, contract)
    if not df.empty:
        return df
    return load_from_dataset(PARQUET_DIR, zone, contract)

df = load_prices(ZONE, CONTRACT)
print('Loaded rows:', len(df))
display(df.tail(5))

Loaded rows: 0


,bidding_zone,contract,ts_utc,price,currency,unit,resolution


In [4]:
# Check latest timestamp vs now (UTC)
from datetime import timezone

EXPECT_MAX_AGE_HOURS = int(os.getenv('WB4U_EXPECT_MAX_AGE_HOURS', '48'))

if df.empty:
    print('No data found. Ensure the updater has run at least once.')
else:
    last_ts = df['ts_utc'].iloc[-1]
    now_utc = dt.datetime.now(tz=timezone.utc)
    age = now_utc - last_ts
    print('Last timestamp:', last_ts)
    print('Now (UTC):     ', now_utc)
    print('Age (hours):   ', round(age.total_seconds()/3600, 2))
    if age.total_seconds() <= EXPECT_MAX_AGE_HOURS*3600:
        print('Status: OK (within', EXPECT_MAX_AGE_HOURS, 'hours)')
    else:
        print('Status: STALE (exceeds', EXPECT_MAX_AGE_HOURS, 'hours)')


No data found. Ensure the updater has run at least once.


In [5]:
# Optional: trigger daily updater for the last 2 days, then re-check
RUN_UPDATE = False  # set True to run
TOKEN = os.getenv('ENTSOE_TOKEN')  # or set here e.g., 'YOUR_TOKEN'

if RUN_UPDATE:
    script = (REPO_ROOT / 'scripts' / 'entsoe_daily_update.py').resolve()
    assert script.exists(), f'Cannot find updater: {script}'
    cmd = [sys.executable, str(script), '--zone', ZONE, '--days', '2', '--contract', CONTRACT, '--parquet-dir', PARQUET_DIR]
    if TOKEN: cmd += ['--token', TOKEN]
    import subprocess, shlex
    print('Running:', ' '.join([shlex.quote(c) for c in cmd]))
    res = subprocess.run(cmd, text=True, capture_output=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f'Update failed with code {res.returncode}')
    # Reload and re-check
    df = load_prices(ZONE, CONTRACT)
    print('Reloaded rows:', len(df))
    display(df.tail(5))
else:
    print('Skipped update (set RUN_UPDATE=True to enable).')


Skipped update (set RUN_UPDATE=True to enable).


In [6]:
# Quick plot (last 7 days if available)
import matplotlib.pyplot as plt

if df.empty:
    print('No data to plot.')
else:
    cutoff = dt.datetime.now(dt.timezone.utc) - dt.timedelta(days=7)
    d = df[df['ts_utc'] >= cutoff]
    if d.empty:
        d = df.tail(200)
    ax = d.set_index('ts_utc')['price'].plot(figsize=(12,4), title=f'ENTSO-E {COUNTRY_TAG} {CONTRACT} — Recent Prices')
    ax.set_ylabel('EUR/MWh')
    plt.show()


No data to plot.


In [7]:
# Show a specific monthly parquet partition in a DataFrame
YEAR = 2025
MONTH = 9  # 1-12

import os, pandas as pd, pathlib
part_path = pathlib.Path(PARQUET_DIR) / f"year={YEAR}" / f"month={MONTH:02d}" / "prices.parquet"
print('Monthly file:', part_path)
if not part_path.exists():
    print('No monthly parquet found for this year/month.')
else:
    df_month = pd.read_parquet(part_path)
    if not df_month.empty:
        df_month['ts_utc'] = pd.to_datetime(df_month['ts_utc'], utc=True)
        # Optional: filter by selected zone/contract
        if 'bidding_zone' in df_month.columns:
            df_month = df_month[df_month['bidding_zone'] == ZONE]
        if 'contract' in df_month.columns:
            df_month = df_month[df_month['contract'] == CONTRACT]
        df_month = df_month.sort_values('ts_utc').reset_index(drop=True)
    display(df_month.head(50) if len(df_month) > 50 else df_month)


Monthly file: C:\Users\20203525\Documents\2025 2026\WB4U\energy-profile-HEMS\data\entsoe\parquet\year=2025\month=09\prices.parquet
No monthly parquet found for this year/month.
